In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

# Load raw data
races = pd.read_csv("../data/raw/races.csv")
drivers = pd.read_csv("../data/raw/drivers.csv")
results = pd.read_csv("../data/raw/results.csv")
lap_times = pd.read_csv("../data/raw/lap_times.csv")
circuits = pd.read_csv("../data/raw/circuits.csv")

print("Raw data loaded")
print(f"lap_times shape: {lap_times.shape}")
print(f"circuits shape: {circuits.shape}")

Raw data loaded
lap_times shape: (575029, 6)
circuits shape: (77, 9)


In [2]:
# STEP 1: Calculate circuit-specific baselines (mean and std lap times)
# These will be used for Z-score normalization

print("="*70)
print("STEP 1: CALCULATE CIRCUIT BASELINES")
print("="*70)

circuit_baselines = {}

for circuit_id in circuits['circuitId'].unique():
    circuit_name = circuits[circuits['circuitId'] == circuit_id]['name'].values[0]
    
    # Get all races at this circuit
    circuit_races = races[races['circuitId'] == circuit_id]
    race_ids = circuit_races['raceId'].values
    
    # Get all lap times at this circuit
    circuit_laps = lap_times[lap_times['raceId'].isin(race_ids)]
    circuit_laps = circuit_laps[circuit_laps['milliseconds'].notna()]
    
    if len(circuit_laps) > 0:
        mean_ms = circuit_laps['milliseconds'].mean()
        std_ms = circuit_laps['milliseconds'].std()
        
        circuit_baselines[circuit_id] = {
            'name': circuit_name,
            'mean': mean_ms,
            'std': std_ms,
            'races_count': len(circuit_races)
        }

print(f"\n✓ Circuit baselines calculated for {len(circuit_baselines)} circuits")
print(f"\nSample circuits (mean lap time in seconds):")
for cid in list(circuit_baselines.keys())[:5]:
    stats = circuit_baselines[cid]
    print(f"  {stats['name']:40s}: {stats['mean']/1000:.2f}s avg (±{stats['std']/1000:.2f}s)")

# Save circuit baselines to JSON for app.py
circuit_baselines_for_json = {}
for cid, stats in circuit_baselines.items():
    circuit_baselines_for_json[stats['name']] = {
        'mean': stats['mean'],
        'std': stats['std'],
        'races_count': stats['races_count']
    }

print("\n✓ Circuit baselines saved (will be used for app.py predictions)")

STEP 1: CALCULATE CIRCUIT BASELINES

✓ Circuit baselines calculated for 41 circuits

Sample circuits (mean lap time in seconds):
  Albert Park Grand Prix Circuit          : 98.99s avg (±65.05s)
  Sepang International Circuit            : 110.05s avg (±108.44s)
  Bahrain International Circuit           : 99.78s avg (±39.03s)
  Circuit de Barcelona-Catalunya          : 88.47s avg (±9.84s)
  Istanbul Park                           : 94.79s avg (±22.64s)

✓ Circuit baselines saved (will be used for app.py predictions)


In [3]:
# STEP 2: Extract features for each driver in each race from first 10 laps

print("="*70)
print("STEP 2: FEATURE ENGINEERING")
print("="*70)

features_list = []

# Group by raceId and driverId
for (race_id, driver_id), group in lap_times.groupby(['raceId', 'driverId']):

    # Early laps (first 10)
    early_laps = group[group['lap'] <= 10].copy()

    # At least 5 laps for meaningful features
    if len(early_laps) < 5:
        continue

    # Calculate features
    laps_completed_early = len(early_laps)
    avg_lap_time = early_laps['milliseconds'].mean()
    lap_time_std = early_laps['milliseconds'].std()
    lap_time_min = early_laps['milliseconds'].min()

    # Pace degradation: (avg of last 3 laps) - (avg of first 3 laps)
    if len(early_laps) > 4:
        first_3 = early_laps.head(3)['milliseconds'].mean()
        last_3 = early_laps.tail(3)['milliseconds'].mean()
        pace_degradation = last_3 - first_3
    else:
        pace_degradation = 0

    # Average position in early laps
    avg_position_early = early_laps['position'].mean()

    features_list.append({
        'raceId': race_id,
        'driverId': driver_id,
        'laps_completed_early': laps_completed_early,
        'avg_lap_time': avg_lap_time,
        'lap_time_std': lap_time_std,
        'lap_time_min': lap_time_min,
        'pace_degradation': pace_degradation,
        'avg_position_early': avg_position_early
    })

features_df = pd.DataFrame(features_list)
print(f"\nFeatures created: {features_df.shape}")
print("\nSample features:")
print(features_df.head())

STEP 2: FEATURE ENGINEERING

Features created: (10556, 8)

Sample features:
   raceId  driverId  laps_completed_early  avg_lap_time  lap_time_std  \
0       1         1                    10       93119.0   5739.717550   
1       1         2                    10       99324.2  20609.149914   
2       1         3                    10       91888.5   3922.798951   
3       1         4                    10       93865.1   5566.302931   
4       1         6                    10       92485.2   4498.719961   

   lap_time_min  pace_degradation  avg_position_early  
0         89488      -6939.666667                10.0  
1         91659     -22912.000000                17.0  
2         89468      -3954.333333                 5.9  
3         91083      -7527.666667                14.1  
4         89923      -5622.000000                 7.9  


In [4]:
# STEP 3: Create circuit-aware features using Z-score normalization
# Instead of raw lap times, use lap times relative to circuit average

print("="*70)
print("STEP 3: ADD CIRCUIT-AWARE LAP TIME FEATURE (Z-SCORE)")
print("="*70)

# Merge with races to get circuitId
features_with_circuit = features_df.merge(races[['raceId', 'circuitId']], on='raceId', how='left')

print(f"\nMerged with races: {features_with_circuit.shape}")
print(f"Unique circuits: {features_with_circuit['circuitId'].nunique()}")

# Get circuit names for reference
features_with_circuit = features_with_circuit.merge(
    circuits[['circuitId', 'name']], 
    on='circuitId', 
    how='left'
)

print("\nCreating Z-score relative lap time feature...")

# NEW FEATURE: avg_lap_time_relative_zscore
# Formula: (driver's avg lap time - circuit average) / circuit std dev
# This tells us: "How many standard deviations away from the circuit average?"

features_with_circuit['avg_lap_time_relative_zscore'] = features_with_circuit.apply(
    lambda row: (
        (row['avg_lap_time'] - circuit_baselines[row['circuitId']]['mean']) / 
        circuit_baselines[row['circuitId']]['std']
    ) if row['circuitId'] in circuit_baselines else 0,
    axis=1
)

print("✓ Z-score relative lap time feature created")
print("\nWhat this means:")
print("  +0.67 = 0.67 std devs SLOWER than circuit average (not great)")
print("  -1.23 = 1.23 std devs FASTER than circuit average (excellent!)")
print("  +2.45 = 2.45 std devs SLOWER than circuit average (very slow)")

# Show examples
print("\nExamples of relative lap times:")
sample_df = features_with_circuit[['name', 'avg_lap_time', 'avg_lap_time_relative_zscore']].head(10).copy()
sample_df['avg_lap_time_sec'] = sample_df['avg_lap_time'] / 1000
sample_df = sample_df[['name', 'avg_lap_time_sec', 'avg_lap_time_relative_zscore']]
print(sample_df.to_string(index=False))

STEP 3: ADD CIRCUIT-AWARE LAP TIME FEATURE (Z-SCORE)

Merged with races: (10556, 9)
Unique circuits: 41

Creating Z-score relative lap time feature...
✓ Z-score relative lap time feature created

What this means:
  +0.67 = 0.67 std devs SLOWER than circuit average (not great)
  -1.23 = 1.23 std devs FASTER than circuit average (excellent!)
  +2.45 = 2.45 std devs SLOWER than circuit average (very slow)

Examples of relative lap times:
                          name  avg_lap_time_sec  avg_lap_time_relative_zscore
Albert Park Grand Prix Circuit           93.1190                     -0.090190
Albert Park Grand Prix Circuit           99.3242                      0.005201
Albert Park Grand Prix Circuit           91.8885                     -0.109106
Albert Park Grand Prix Circuit           93.8651                     -0.078720
Albert Park Grand Prix Circuit           92.4852                     -0.099933
Albert Park Grand Prix Circuit           97.0215                     -0.030198
Albert P

In [5]:
# STEP 4: Merge with race results to get actual finish position

race_results = results[['raceId', 'driverId', 'positionText', 'points']].copy()

# Convert positions to numeric (handle non-finishers)
race_results['position_numeric'] = pd.to_numeric(
    race_results['positionText'], errors='coerce'
)

# Remove DNFs (position = NaN)
race_results = race_results[race_results['position_numeric'].notna()].copy()

print(f"Race Results (finishers only): {race_results.shape}")
print(race_results.head())

Race Results (finishers only): (15575, 5)
   raceId  driverId positionText  points  position_numeric
0      18         1            1    10.0               1.0
1      18         2            2     8.0               2.0
2      18         3            3     6.0               3.0
3      18         4            4     5.0               4.0
4      18         5            5     4.0               5.0


In [6]:
# STEP 5: Final merge and create target variables

print("="*70)
print("STEP 5: MERGE AND CREATE TARGETS")
print("="*70)

# Merge features with results
df = features_with_circuit.merge(race_results, on=['raceId', 'driverId'], how='inner')

print(f"\nMerged dataset: {df.shape}")
print(f"Samples: {len(df)}")

# Create target variables
df['top_10'] = (df['position_numeric'] <= 10).astype(int)
df['podium'] = (df['position_numeric'] <= 3).astype(int)
df['winner'] = (df['position_numeric'] == 1).astype(int)
df['runner_up'] = (df['position_numeric'] == 2).astype(int)
df['third'] = (df['position_numeric'] == 3).astype(int)

print(f"\nTarget distribution:")
print(f"  Top 10 finishes:  {df['top_10'].sum():5d} ({100*df['top_10'].mean():.1f}%)")
print(f"  Podium (top 3):   {df['podium'].sum():5d} ({100*df['podium'].mean():.1f}%)")
print(f"  Winners (1st):    {df['winner'].sum():5d} ({100*df['winner'].mean():.1f}%)")
print(f"  Runner-up (2nd):  {df['runner_up'].sum():5d} ({100*df['runner_up'].mean():.1f}%)")
print(f"  Third place:      {df['third'].sum():5d} ({100*df['third'].mean():.1f}%)")

print("\nFinal dataset:")
print(df[['name', 'avg_lap_time', 'avg_lap_time_relative_zscore', 'position_numeric', 'top_10']].head())

STEP 5: MERGE AND CREATE TARGETS

Merged dataset: (8468, 14)
Samples: 8468

Target distribution:
  Top 10 finishes:   5270 (62.2%)
  Podium (top 3):    1590 (18.8%)
  Winners (1st):      530 (6.3%)
  Runner-up (2nd):    530 (6.3%)
  Third place:        530 (6.3%)

Final dataset:
                             name  avg_lap_time  avg_lap_time_relative_zscore  \
0  Albert Park Grand Prix Circuit       99324.2                      0.005201   
1  Albert Park Grand Prix Circuit       91888.5                     -0.109106   
2  Albert Park Grand Prix Circuit       93865.1                     -0.078720   
3  Albert Park Grand Prix Circuit       97021.5                     -0.030198   
4  Albert Park Grand Prix Circuit       94468.3                     -0.069448   

   position_numeric  top_10  
0              10.0       1  
1               6.0       1  
2               5.0       1  
3               8.0       1  
4              15.0       0  


In [7]:
# STEP 6: Statistics and analysis

print("="*70)
print("STEP 6: STATISTICS AND ANALYSIS")
print("="*70)

print("\nFeature statistics:")
print(df[[
    'laps_completed_early', 
    'avg_lap_time', 
    'avg_lap_time_relative_zscore',  # NEW: relative lap time
    'lap_time_std', 
    'lap_time_min', 
    'pace_degradation', 
    'avg_position_early',
    'top_10'
]].describe())

# Analysis of relative lap time feature
print("\n" + "="*70)
print("Z-SCORE RELATIVE LAP TIME ANALYSIS")
print("="*70)

print(f"\nRelative lap time range: {df['avg_lap_time_relative_zscore'].min():.2f} to {df['avg_lap_time_relative_zscore'].max():.2f}")
print(f"Mean: {df['avg_lap_time_relative_zscore'].mean():.4f} (should be ~0)")
print(f"Std: {df['avg_lap_time_relative_zscore'].std():.4f} (should be ~1 per circuit)")

print("\nInterpretation:")
print(f"  Negative z-scores (faster than average): {(df['avg_lap_time_relative_zscore'] < 0).sum()} drivers")
print(f"  Positive z-scores (slower than average): {(df['avg_lap_time_relative_zscore'] > 0).sum()} drivers")

# Top-10 rate by relative performance
print("\nTop-10 finish rate by relative performance:")
for threshold in [-2, -1, 0, 1, 2]:
    subset = df[df['avg_lap_time_relative_zscore'] <= threshold]
    if len(subset) > 0:
        top10_pct = 100 * subset['top_10'].mean()
        print(f"  Z-score ≤ {threshold:2d}: {len(subset):5d} drivers, {top10_pct:5.1f}% finish top-10")

STEP 6: STATISTICS AND ANALYSIS

Feature statistics:
       laps_completed_early   avg_lap_time  avg_lap_time_relative_zscore  \
count           8468.000000    8468.000000                   8468.000000   
mean               9.999173  105303.161573                      0.208825   
std                0.058518   42523.536608                      0.709349   
min                5.000000   69568.000000                     -1.443195   
25%               10.000000   86827.575000                     -0.097234   
50%               10.000000   97455.450000                     -0.008331   
75%               10.000000  109612.975000                      0.233263   
max               10.000000  479358.600000                      5.144526   

       lap_time_std   lap_time_min  pace_degradation  avg_position_early  \
count  8.468000e+03    8468.000000      8.468000e+03         8468.000000   
mean   2.604211e+04   92922.429499     -1.757306e+04           10.133506   
std    1.103184e+05   12888.182663

In [8]:
# Circuit breakdown
print("\nCircuit breakdown (top 10 by number of races):")

circuit_breakdown = df.groupby('name').agg({
    'top_10': ['count', 'sum', 'mean'],
    'avg_lap_time': 'mean',
    'avg_lap_time_relative_zscore': ['mean', 'std']
}).round(3)

circuit_breakdown.columns = ['races', 'top10_count', 'top10_rate', 'avg_lap_time_ms', 'rel_zscore_mean', 'rel_zscore_std']
circuit_breakdown['avg_lap_time_sec'] = (circuit_breakdown['avg_lap_time_ms'] / 1000).round(2)
circuit_breakdown = circuit_breakdown.sort_values('races', ascending=False)

print(circuit_breakdown[['races', 'top10_rate', 'avg_lap_time_sec', 'rel_zscore_mean', 'rel_zscore_std']].head(10))


Circuit breakdown (top 10 by number of races):
                                races  top10_rate  avg_lap_time_sec  \
name                                                                  
Circuit de Barcelona-Catalunya    470       0.609             91.87   
Silverstone Circuit               470       0.615            130.97   
Hungaroring                       461       0.607             94.87   
Autodromo Nazionale di Monza      454       0.615             93.48   
Autódromo José Carlos Pace        422       0.637             93.75   
Suzuka Circuit                    416       0.601            124.95   
Circuit de Monaco                 403       0.685            100.11   
Circuit de Spa-Francorchamps      396       0.626            130.25   
Circuit Gilles Villeneuve         383       0.674             88.51   
Bahrain International Circuit     369       0.569            109.67   

                                rel_zscore_mean  rel_zscore_std  
name                             

In [9]:
# STEP 8: Save everything for next notebooks

print("="*70)
print("STEP 8: SAVE DATA FOR NEXT STEPS")
print("="*70)

# Save the main dataframe (will be re-created in notebook 03, but keeping for reference)
# df.to_csv('../data/processed/features_engineered.csv', index=False)
# print("\n✓ Features saved to CSV")

# Save circuit baselines for app.py
import json
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)

with open("../data/processed/circuit_baselines.json", "w") as f:
    json.dump(circuit_baselines_for_json, f, indent=2)

print("\n✓ Circuit baselines saved to circuit_baselines.json")
print("  (Will be used by app.py for real-time Z-score calculations)")

# Display what was saved
print("\nSample circuit baselines (for app.py):")
for circuit_name in list(circuit_baselines_for_json.keys())[:3]:
    stats = circuit_baselines_for_json[circuit_name]
    print(f"  {circuit_name:40s}: mean={stats['mean']/1000:6.2f}s, std={stats['std']/1000:5.2f}s")

print("\n" + "="*70)
print("FEATURE ENGINEERING COMPLETE!")
print("="*70)
print(f"\nDataset ready for notebook 03:")
print(f"  Total samples: {len(df)}")
print(f"  Features per sample: 7 (including new Z-score relative lap time)")
print(f"  Targets available: top_10, podium, winner, runner_up, third")
print(f"\nKey improvement:")
print(f"  OLD: Raw lap time (95s looks same on Monaco and Monza)")
print(f"  NEW: Relative Z-score (+0.67 on Monaco vs +6.5 on Monza) ✓")

STEP 8: SAVE DATA FOR NEXT STEPS

✓ Circuit baselines saved to circuit_baselines.json
  (Will be used by app.py for real-time Z-score calculations)

Sample circuit baselines (for app.py):
  Albert Park Grand Prix Circuit          : mean= 98.99s, std=65.05s
  Sepang International Circuit            : mean=110.05s, std=108.44s
  Bahrain International Circuit           : mean= 99.78s, std=39.03s

FEATURE ENGINEERING COMPLETE!

Dataset ready for notebook 03:
  Total samples: 8468
  Features per sample: 7 (including new Z-score relative lap time)
  Targets available: top_10, podium, winner, runner_up, third

Key improvement:
  OLD: Raw lap time (95s looks same on Monaco and Monza)
  NEW: Relative Z-score (+0.67 on Monaco vs +6.5 on Monza) ✓
